# R2AI Financial Table Q&A — PIPELINE RUNNER (Google Colab GPU T4)
Notebook trung tâm điều phối toàn bộ quy trình End-to-End của dự án trên **Google Colab** (GPU T4 / A100).

### Hai Lựa Chọn Luồng Dữ Liệu:
* **Lựa chọn A (Khuyên dùng - Tiết kiệm thời gian):** Nạp nhanh kho dữ liệu/chỉ mục từ `data_backup.zip` trên Google Drive rồi chạy thẳng suy luận (`infer`).
* **Lựa chọn B (Build từ đầu - Full Pipeline):** Chạy tuần tự từng bước: **Fetch (Tải BCTC) $\rightarrow$ Corpus (Trích xuất bảng) $\rightarrow$ Index (Tạo chỉ mục) $\rightarrow$ Infer (Suy luận) $\rightarrow$ Package (Đóng gói) $\rightarrow$ Evaluate (Đánh giá)**.

## Mục 1: Khởi tạo & Cài đặt Môi trường trên Colab

In [ ]:
# 1.1. Nhận diện Môi trường, Mount Google Drive và Đồng bộ Repo vào ổ SSD NVMe của Colab
import os, sys, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
IN_KAGGLE = "kaggle_web_client" in sys.modules or "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists()
IN_LOCAL = not IN_COLAB and not IN_KAGGLE

print(f"Môi trường hiện tại: {'Google Colab' if IN_COLAB else ('Kaggle' if IN_KAGGLE else 'Local')}")

# 1. Mount Google Drive
if IN_COLAB and not Path("/content/drive").exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f"Mount Drive: {e}")

# 2. Đảm bảo mã nguồn nằm trên ổ SSD NVMe tốc độ cao của Colab
local_repo = Path("/content/R2AI-Stage-2")
drive_repo = Path("/content/drive/MyDrive/R2AI-Stage-2")

if IN_COLAB:
    if not local_repo.exists():
        if drive_repo.exists():
            print("Đang sao chép mã nguồn từ Drive sang ổ SSD NVMe Colab...")
            shutil.copytree(drive_repo, local_repo, ignore=shutil.ignore_patterns("data", ".git", ".venv", "__pycache__"))
        else:
            print("Đang clone repository từ GitHub...")
            !git clone https://github.com/Nostagi/R2AI-Stage-2.git /content/R2AI-Stage-2

    local_repo.mkdir(parents=True, exist_ok=True)
    %cd /content/R2AI-Stage-2

print(f"Thư mục làm việc hiện tại: {os.getcwd()}")

In [ ]:
# 1.2. Cài đặt toàn bộ dependencies từ requirements.txt
print("Đang cài đặt dependencies...")
!pip uninstall -y -q torchaudio
!pip install -q -r requirements.txt
print("Đã cài đặt xong môi trường chuẩn!")

In [ ]:
# 1.3. Kiểm tra GPU CUDA
import torch
print(f"CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Thiết bị GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB VRAM)")
else:
    print("CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU!")

--- 
## Mục 2: Chuẩn bị Dữ liệu & Chỉ mục (CHỌN PHƯƠNG ÁN 2.A HOẶC 2.B)

> * Nếu bạn **ĐÃ CÓ** file `data_backup.zip` trên Drive: Hãy chạy **Mục 2.A**.
> * Nếu bạn **CHƯA CÓ** dữ liệu hoặc muốn **BUILD LẠI TỪ ĐẦU**: Bỏ qua 2.A và chạy **Mục 2.B**.

### Phương án 2.A: Nạp nhanh từ file sao lưu `data_backup.zip` (Tự động chống lồng thư mục data/data)

In [ ]:
# 2.A. Giải nén data_backup.zip từ Google Drive vào ổ NVMe (/content/R2AI-Stage-2/data)
import zipfile, shutil
from pathlib import Path

repo_root = Path("/content/R2AI-Stage-2")
data_dir = repo_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

drive_backup = Path("/content/drive/MyDrive/backup/data_backup.zip")
if not drive_backup.exists():
    drive_backup = Path("/content/drive/MyDrive/data_backup.zip")

if drive_backup.exists():
    print(f"Tìm thấy data backup tại: {drive_backup}")
    print("Đang giải nén vào ổ SSD NVMe...")
    with zipfile.ZipFile(drive_backup, 'r') as zf:
        has_data_prefix = any(n.startswith("data/") or n.startswith("data\\") for n in zf.namelist())
        extract_target = repo_root if has_data_prefix else data_dir
        zf.extractall(extract_target)
    
    # Dọn dẹp nếu bị lồng data/data
    nested_data = data_dir / "data"
    if nested_data.exists() and nested_data.is_dir():
        print("Phát hiện thư mục lồng data/data/ -> Đang tự động chuyển về data/...")
        for item in nested_data.iterdir():
            dest = data_dir / item.name
            if dest.exists():
                if dest.is_dir(): shutil.rmtree(dest)
                else: dest.unlink()
            shutil.move(str(item), str(dest))
        nested_data.rmdir()
        
    print("Giải nén hoàn tất!")
    # Kiểm tra tính toàn vẹn
    !python main.py verify
else:
    print("Không tìm thấy data_backup.zip trên Drive. Vui lòng chuyển sang chạy Phương án 2.B bên dưới.")

### Phương án 2.B: Xây dựng toàn bộ Kho dữ liệu & Chỉ mục từ đầu (Full Pipeline Build)

In [ ]:
# 2.B.0 (Stage 00): Tải dữ liệu BCTC thô từ Hugging Face
!python main.py fetch

In [ ]:
# 2.B.1 (Stage 01): Bóc tách OCR, ghép bảng gãy & chuẩn hóa sang 119k file CSV (data/processed/)
!python main.py corpus

In [ ]:
# 2.B.2 (Stage 02): Xây dựng chỉ mục tìm kiếm (BM25 + FAISS Dense BGE-M3 + MetadataStore)
!python main.py index

In [ ]:
# 2.B.3. Kiểm tra tính toàn vẹn của Corpus và Index sau khi build
!python main.py verify

In [ ]:
# 2.B.4 (Tùy chọn): Đóng gói thư mục data/ thành data_backup.zip và sao lưu vào Google Drive
print("Đang nén toàn bộ thư mục data/ để sao lưu lên Google Drive...")
!mkdir -p /content/drive/MyDrive/backup
!zip -q -r /content/drive/MyDrive/backup/data_backup.zip data/
print("Đã lưu data_backup.zip lên Google Drive: /content/drive/MyDrive/backup/data_backup.zip")

---
## Mục 3: Suy Luận Theo Kiến Trúc 2 Pha Tách Rời (Hoặc Chạy Toàn Bộ)

Bạn có thể chọn một trong hai luồng thực thi:
* **Luồng Khuyên Dùng (2 Pha Tách Rời):** Chạy **Mục 3.A** (Retrieval ~1-2 phút, xuất `retrieval_results.json` & đóng gói `retrieval_submission.zip` vào Google Drive để nộp kiểm thử nhanh F2) $\rightarrow$ sau đó chạy **Mục 3.B** (Generation & Execution ~20 phút với cơ chế Checkpoint Resume từ Google Drive).
* **Luồng Trực Tiếp (All-in-One):** Bỏ qua 3.A và 3.B, chạy thẳng **Mục 3.C** (Chạy toàn bộ 2 pha liền mạch với Checkpoint Resume).


### Mục 3.A: Chạy Pha 1 (Retrieval Pipeline & Đóng Gói Nộp Kiểm Thử Nhanh F2)


In [ ]:
# 3.A.1. Tạo thư mục output & Chạy Pha 1 (Retrieval độc lập trên toàn bộ câu hỏi)
!mkdir -p outputs/retrieval /content/drive/MyDrive/backup
!python -u main.py retrieve \
    --questions data/questions/questions.jsonl \
    --out outputs/retrieval/retrieval_results.json

In [ ]:
# 3.A.2. (Tùy chọn) Đóng gói file nộp kiểm thử nhanh DOCS_F2 và TABLES_F2 lưu thẳng vào Google Drive
!python main.py package \
    --pred outputs/retrieval/retrieval_results.json \
    --name retrieval_submission \
    --out /content/drive/MyDrive/backup/retrieval_submission.zip

print('Đã đóng gói xong retrieval_submission.zip tại Google Drive sẵn sàng nộp kiểm thử F2!')

### Mục 3.B: Chạy Pha 2 (Generation & Execution Pipeline từ kết quả Pha 1 - Có Checkpoint Resume từ Google Drive)


In [ ]:
# 3.B.1. Nạp file retrieval trung gian & Khôi phục Checkpoint cho Pha 2 từ Google Drive (nếu có)
import json, os, shutil
from pathlib import Path

!mkdir -p outputs/predictions outputs/retrieval /content/drive/MyDrive/backup

retrieval_file = Path('outputs/retrieval/retrieval_results.json')
local_pred = Path('outputs/predictions/final_results.json')
backup_pred = Path('/content/drive/MyDrive/backup/final_results.json')

# 1. Khôi phục retrieval_results.json nếu chưa có (tìm trong Google Drive)
if not retrieval_file.exists():
    backup_ret = Path('/content/drive/MyDrive/backup/retrieval_results.json')
    if backup_ret.exists():
        shutil.copy2(backup_ret, retrieval_file)
        print(f'Đã khôi phục retrieval_results.json từ {backup_ret}')

if not retrieval_file.exists():
    raise FileNotFoundError('Chưa có outputs/retrieval/retrieval_results.json! Hãy chạy Mục 3.A trước.')

# 2. Khôi phục checkpoint final_results.json từ Google Drive
if backup_pred.exists() and not local_pred.exists():
    print(f'Tìm thấy checkpoint tại {backup_pred} -> Đang khôi phục để resume Phase 2...')
    shutil.copy2(backup_pred, local_pred)

if local_pred.exists():
    try:
        with open(local_pred, 'r', encoding='utf-8') as f:
            data = json.load(f)
            print(f'Sẵn sàng RESUME Pha 2: Đã hoàn thành {len(data)}/1012 câu trong checkpoint!')
    except Exception as e:
        print(f'Lỗi đọc checkpoint: {e}')
else:
    print('Chưa có checkpoint Pha 2 -> Sẽ chạy từ câu đầu tiên (Q1).')

# 3.B.2. Chạy Text-to-Pandas & Execution với mô hình Qwen2.5-Coder-7B-Instruct
# Tự động lưu checkpoint mỗi 5 câu và đồng bộ sang /content/drive/MyDrive/backup/final_results.json
!python -u main.py generate \
    --retrieval outputs/retrieval/retrieval_results.json \
    --model Qwen/Qwen2.5-Coder-7B-Instruct \
    --backend transformers \
    --pred outputs/predictions/final_results.json


### Mục 3.C: Chạy Toàn Bộ 2 Pha Liền Mạch (Full Inference End-to-End - Có Checkpoint Resume từ Google Drive)


In [ ]:
# 3.C.1. Khôi phục Checkpoint từ Google Drive nếu có
import json, os, shutil
from pathlib import Path

!mkdir -p outputs/predictions /content/drive/MyDrive/backup

local_pred = Path('outputs/predictions/questions_pred.json')
backup_pred = Path('/content/drive/MyDrive/backup/questions_pred.json')

if backup_pred.exists() and not local_pred.exists():
    print(f'Tìm thấy checkpoint tại {backup_pred} -> Đang khôi phục để resume...')
    shutil.copy2(backup_pred, local_pred)

if local_pred.exists():
    try:
        with open(local_pred, 'r', encoding='utf-8') as f:
            data = json.load(f)
            print(f'Sẵn sàng RESUME: Đã có {len(data)}/1012 câu trong checkpoint!')
    except Exception as e:
        print(f'Lỗi đọc checkpoint: {e}')
else:
    print('Chưa có checkpoint cũ -> Sẽ chạy suy luận toàn bộ từ câu đầu tiên (Q1).')

# 3.C.2. Chạy suy luận liền mạch toàn bộ 1,012 câu hỏi với mô hình Qwen2.5-Coder-7B-Instruct
# Tự động lưu checkpoint mỗi 5 câu và đồng bộ sang /content/drive/MyDrive/backup/questions_pred.json
!python -u main.py infer \
    --questions data/questions/questions.jsonl \
    --model Qwen/Qwen2.5-Coder-7B-Instruct \
    --backend transformers \
    --pred outputs/predictions/questions_pred.json


---
## Mục 4: Đóng Gói File Nộp Bài Hoàn Chỉnh (Stage 04 Package)


In [ ]:
# 4. Tự động nhận diện kết quả từ Mục 3.B (final_results.json) hoặc Mục 3.C (questions_pred.json) để đóng gói
import shutil
from pathlib import Path

pred_candidates = [
    Path('outputs/predictions/final_results.json'),
    Path('outputs/predictions/questions_pred.json'),
]
target_pred = None
for p in pred_candidates:
    if p.exists():
        target_pred = p
        break

if target_pred is None:
    raise FileNotFoundError('Chưa tìm thấy file dự đoán trong outputs/predictions/! Hãy chạy Mục 3 trước.')

print(f'Đang đóng gói file nộp bài từ: {target_pred}')
!python main.py package \
    --pred {target_pred} \
    --name submission \
    --out /content/drive/MyDrive/backup/submission.zip

if Path('/content/drive/MyDrive/backup').exists():
    shutil.copy2(target_pred, f'/content/drive/MyDrive/backup/{target_pred.name}')
print(f'Đã lưu submission.zip và {target_pred.name} tại Google Drive sẵn sàng tải về nộp bài!')


---
## Mục 5: Đánh Giá Điểm Số (Stage 05 Evaluate)


In [ ]:
# 5. Đánh giá điểm số trên tập nhãn chuẩn gold.json nếu có
if Path('labels/gold.json').exists():
    eval_pred = target_pred if 'target_pred' in locals() and target_pred and target_pred.exists() else Path('outputs/predictions/final_results.json')
    !python main.py evaluate \
        --pred {eval_pred} \
        --gold labels/gold.json
else:
    print('Chưa có file nhãn chuẩn labels/gold.json -> Bỏ qua bước evaluate.')
